In [1]:
# !pip install datasets

In [2]:
# 패키지 준비

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset


In [ ]:
# 데이터 준비

# NSMC (네이버 영화 리뷰 감성 분석) 데이터 활용 예시
# 실제 사용 시: https://github.com/e9t/nsmc 에서 다운로드
# 또는 datasets 라이브러리: pip install datasets

# nsmc = load_dataset("nsmc")
# DatasetDict({
#     train: Dataset({features: ['id', 'document', 'label'], num_rows: 150000}),
#     test:  Dataset({features: ['id', 'document', 'label'], num_rows: 50000})
# })

nsmc = load_dataset(
    "csv",
    data_files={
        "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
    },
    sep="\t"
)

# 데이터 확인
# print(nsmc)
# print(nsmc['train'].column_names)
# print(nsmc['train'][0])
# print(nsmc['train'][0]['document'], nsmc['train'][0]['label'])

class NSMCDataset(Dataset):
    """네이버 영화 리뷰 감성 분류 데이터셋"""
    
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data[idx]
        text = str(row['document']) if row['document'] else ""
        label = int(row['label'])
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length, # 모든 문장은 128개 토큰으로 구성
            padding='max_length',
            truncation=True,
            return_tensors='pt' # pytorch 형식으로 반환
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 데이터로더 생성
MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NSMCDataset(nsmc['train'], tokenizer)
test_dataset  = NSMCDataset(nsmc['test'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4)

print(f"훈련: {len(train_dataset):,}개 / 테스트: {len(test_dataset):,}개")

print(train_dataset[0])

훈련: 150,000개 / 테스트: 50,000개
{'input_ids': tensor([   2, 1376,  831, 2604,   18,   18, 4229, 9801, 2075, 2203, 2182, 4243,
           3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
  

In [4]:
# 모델 설계 

import torch.nn as nn
from transformers import AutoModel

class BERTClassifier(nn.Module):
    """BERT 기반 이진 분류 모델"""
    
    def __init__(self, model_name: str, num_classes: int, dropout_rate: float = 0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name) # 언어 (이해) 모델 준비
        hidden_size = self.bert.config.hidden_size  # 768
        
        self.classifier = nn.Sequential( # 분류 모델 준비
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, input_ids, attention_mask):
        # BERT 인코딩
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # [CLS] 토큰 표현 추출 -> 문장 임베딩 추출
        cls_output = outputs.last_hidden_state[:, 0, :]  # (batch, 768)
        # 분류 헤드 통과
        logits = self.classifier(cls_output)              # (batch, num_classes)
        return logits

# 모델 초기화
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(model_name=MODEL_NAME, num_classes=2).to(device)
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")
# 약 110,000,000 (1.1억)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 파라미터: 110,618,882


In [5]:
# 학습 설계
from torch.optim import AdamW

# BERT 레이어는 낮은 lr, 분류기는 높은 lr
# → 사전학습 가중치를 보존하면서 분류기를 빠르게 학습

optimizer = AdamW([
    {'params': model.bert.parameters(),        'lr': 2e-5},  # BERT: 낮은 lr
    {'params': model.classifier.parameters(),  'lr': 1e-4},  # Classifier: 높은 lr
], weight_decay=0.01)


In [ ]:
# 학습 루프

from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

# 스케줄러: warmup + linear decay
NUM_EPOCHS = 3
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = total_steps // 10  # 전체의 10%를 warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        
        # 그래디언트 클리핑 (BERT 훈련 시 중요)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().tolist())
    
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

# 전체 훈련 루프
best_acc = 0.0
print(f"{'Epoch':>5} {'Train Loss':>10} {'Train Acc':>9} {'Val Loss':>8} {'Val Acc':>7}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, device)
    
    print(f"{epoch:>5} {train_loss:>10.4f} {train_acc:>9.4f} {val_loss:>8.4f} {val_acc:>7.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_bert_nsmc.pt")
        print(f"  ✓ 최고 모델 저장 (val_acc={val_acc:.4f})")


Epoch Train Loss Train Acc Val Loss Val Acc
--------------------------------------------------


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 베스트 모델 로드 후 최종 평가
model.load_state_dict(torch.load("best_bert_nsmc.pt", map_location=device))
_, final_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, device)

# 분류 리포트
label_names = ["부정(0)", "긍정(1)"]
print(classification_report(all_labels, all_preds, target_names=label_names))
# 예시 출력:
#               precision    recall  f1-score   support
#   부정(0)       0.91      0.89      0.90     25000
#   긍정(1)       0.89      0.91      0.90     25000
#   accuracy                           0.90     50000

# 혼동 행렬
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix - NSMC Sentiment")
plt.ylabel("실제 레이블")
plt.xlabel("예측 레이블")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
